# Positions Dataset

The positions file is a gzipped JSONL file, so each line is one JSON object representing one chess position.

Each record contains:
- `game_id`: source game identifier or file name
- `white`, `black`: player names from the PGN headers
- `date`: PGN date string
- `ply`: 0-based ply index inside the game
- `side_to_move`: `white` or `black`
- `fen`: full board state in FEN form
- `legal_moves`: list of legal moves in UCI format for that position
- `chosen_move`: the move played from that position when the parser knew it was your turn, otherwise `null`
- `white_clock`, `black_clock`: clock values when available, otherwise `null`
- `move_number`: fullmove number from the FEN

The raw file includes every position from every parsed game, so many rows will have `chosen_move = null`. For training, we filter to `my_positions = positions[positions["chosen_move"].notna()]` and only use that labeled subset.

In [11]:
from pathlib import Path
import gzip

import pandas as pd
import chess

POSITIONS_PATH = Path(r"C:\Users\srina\Coding\Chess-Bot\lichess-bot\game-data\positions.jsonl.gz")

with gzip.open(POSITIONS_PATH, "rt", encoding="utf-8") as f:
    positions = pd.read_json(f, lines=True)

my_positions = positions[positions["chosen_move"].notna()].copy()

print(f"Total positions: {len(positions):,}")
print(f"Positions with chosen_move: {len(my_positions):,}")
print(f"Coverage: {len(my_positions) / len(positions):.2%}")
my_positions.head()

Total positions: 739,764
Positions with chosen_move: 370,314
Coverage: 50.06%


,game_id,white,black,date,ply,side_to_move,fen,legal_moves,chosen_move,white_clock,black_clock,move_number
0,1,ChessCHNC,Coach-David,2026-04-04,0,white,rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w ...,"[g1h3, g1f3, b1c3, b1a3, h2h3, g2g3, f2f3, e2e...",e2e4,NaN,NaN,1
2,1,ChessCHNC,Coach-David,2026-04-04,2,white,rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBN...,"[g1h3, g1f3, g1e2, f1a6, f1b5, f1c4, f1d3, f1e...",g1f3,NaN,NaN,2
4,1,ChessCHNC,Coach-David,2026-04-04,4,white,rnbqkb1r/pppp1ppp/5n2/4p3/4P3/5N2/PPPP1PPP/RNB...,"[f3g5, f3e5, f3h4, f3d4, f3g1, h1g1, f1a6, f1b...",d2d4,NaN,NaN,3
6,1,ChessCHNC,Coach-David,2026-04-04,6,white,r1bqkb1r/pppp1ppp/2n2n2/4p3/3PP3/5N2/PPP2PPP/R...,"[f3g5, f3e5, f3h4, f3d2, f3g1, h1g1, f1a6, f1b...",d4e5,NaN,NaN,4
8,1,ChessCHNC,Coach-David,2026-04-04,8,white,r1bqkb1r/pppp1ppp/2n5/4P3/4n3/5N2/PPP2PPP/RNBQ...,"[f3g5, f3h4, f3d4, f3d2, f3g1, h1g1, f1a6, f1b...",f1d3,NaN,NaN,5


In [12]:
my_positions.info()

<class 'pandas.DataFrame'>
Index: 370314 entries, 0 to 739763
Data columns (total 12 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   game_id       370314 non-null  int64         
 1   white         370314 non-null  str           
 2   black         370314 non-null  str           
 3   date          370314 non-null  datetime64[us]
 4   ply           370314 non-null  int64         
 5   side_to_move  370314 non-null  str           
 6   fen           370314 non-null  str           
 7   legal_moves   370314 non-null  object        
 8   chosen_move   370314 non-null  str           
 9   white_clock   0 non-null       float64       
 10  black_clock   0 non-null       float64       
 11  move_number   370314 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(3), object(1), str(5)
memory usage: 36.7+ MB


## Feature Engineering Strategy

Short answer: `python-chess` gives you everything you need directly from FEN.

Useful APIs:
- `chess.Board(fen)` to load position
- `board.piece_map()` for square -> piece dictionary
- `board.pieces(piece_type, color)` for bitboard-like square sets
- `board.occupied`, `board.occupied_co[color]` for occupancy bitboards
- `board.legal_moves` for candidate moves
- `board.is_capture(move)`, `board.gives_check(move)`, `board.is_castling(move)` for move properties

A practical setup:
1. Position tensor features (12x8x8 piece planes)
2. Position scalar features (material, mobility, castling rights, checks, etc.)
3. Move features (capture/check/promo/castle, from/to squares, moving piece)
4. Delta features = position-after-move minus position-before-move

Start with lightweight handcrafted features first, then optionally add tensor inputs for a model later.

In [13]:
import numpy as np

PIECE_VALUES = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0,
}

PIECE_ORDER = [
    (chess.WHITE, chess.PAWN),
    (chess.WHITE, chess.KNIGHT),
    (chess.WHITE, chess.BISHOP),
    (chess.WHITE, chess.ROOK),
    (chess.WHITE, chess.QUEEN),
    (chess.WHITE, chess.KING),
    (chess.BLACK, chess.PAWN),
    (chess.BLACK, chess.KNIGHT),
    (chess.BLACK, chess.BISHOP),
    (chess.BLACK, chess.ROOK),
    (chess.BLACK, chess.QUEEN),
    (chess.BLACK, chess.KING),
]


def board_to_piece_planes(board: chess.Board) -> np.ndarray:
    """Return 12x8x8 binary tensor [piece_plane, rank, file]."""
    planes = np.zeros((12, 8, 8), dtype=np.int8)
    for i, (color, piece_type) in enumerate(PIECE_ORDER):
        for sq in board.pieces(piece_type, color):
            rank = chess.square_rank(sq)
            file = chess.square_file(sq)
            planes[i, rank, file] = 1
    return planes


def board_bitboards(board: chess.Board) -> dict:
    """Expose occupancy and piece bitboards as integers."""
    out = {
        "occupied": int(board.occupied),
        "occupied_white": int(board.occupied_co[chess.WHITE]),
        "occupied_black": int(board.occupied_co[chess.BLACK]),
    }
    for color_name, color in [("w", chess.WHITE), ("b", chess.BLACK)]:
        for pt_name, pt in [("p", chess.PAWN), ("n", chess.KNIGHT), ("b", chess.BISHOP), ("r", chess.ROOK), ("q", chess.QUEEN), ("k", chess.KING)]:
            mask = 0
            for sq in board.pieces(pt, color):
                mask |= 1 << sq
            out[f"{color_name}_{pt_name}"] = int(mask)
    return out


def material_score(board: chess.Board) -> int:
    w = 0
    b = 0
    for pt, val in PIECE_VALUES.items():
        w += len(board.pieces(pt, chess.WHITE)) * val
        b += len(board.pieces(pt, chess.BLACK)) * val
    return w - b


def position_features(board: chess.Board) -> dict:
    turn = 1 if board.turn == chess.WHITE else 0
    wk = board.king(chess.WHITE)
    bk = board.king(chess.BLACK)

    # Mobility from side to move perspective.
    mobility = board.legal_moves.count()

    return {
        "turn_white": turn,
        "fullmove_number": board.fullmove_number,
        "halfmove_clock": board.halfmove_clock,
        "is_check": int(board.is_check()),
        "legal_move_count": mobility,
        "material_diff": material_score(board),
        "white_can_castle_k": int(board.has_kingside_castling_rights(chess.WHITE)),
        "white_can_castle_q": int(board.has_queenside_castling_rights(chess.WHITE)),
        "black_can_castle_k": int(board.has_kingside_castling_rights(chess.BLACK)),
        "black_can_castle_q": int(board.has_queenside_castling_rights(chess.BLACK)),
        "white_king_square": int(wk) if wk is not None else -1,
        "black_king_square": int(bk) if bk is not None else -1,
    }


def move_features(board: chess.Board, move: chess.Move) -> dict:
    piece = board.piece_at(move.from_square)
    return {
        "uci": move.uci(),
        "from_square": int(move.from_square),
        "to_square": int(move.to_square),
        "from_file": chess.square_file(move.from_square),
        "from_rank": chess.square_rank(move.from_square),
        "to_file": chess.square_file(move.to_square),
        "to_rank": chess.square_rank(move.to_square),
        "is_capture": int(board.is_capture(move)),
        "is_check": int(board.gives_check(move)),
        "is_castle": int(board.is_castling(move)),
        "is_en_passant": int(board.is_en_passant(move)),
        "is_promotion": int(move.promotion is not None),
        "promotion_piece": int(move.promotion) if move.promotion else 0,
        "piece_type": int(piece.piece_type) if piece else 0,
    }


def delta_features(board: chess.Board, move: chess.Move) -> dict:
    before = position_features(board)
    b2 = board.copy(stack=False)
    b2.push(move)
    after = position_features(b2)
    return {f"delta_{k}": after[k] - before[k] for k in before.keys() if isinstance(before[k], (int, float))}


def make_labeled_examples(row, negative_samples: int = 7, rng_seed: int = 7):
    """Create one positive + sampled negatives from a labeled my_positions row."""
    board = chess.Board(row.fen)
    chosen = chess.Move.from_uci(row.chosen_move)
    legal = list(board.legal_moves)
    legal_uci = [m.uci() for m in legal]

    if chosen.uci() not in legal_uci:
        return []

    rng = np.random.default_rng(rng_seed)
    neg_pool = [m for m in legal if m.uci() != chosen.uci()]
    if len(neg_pool) > negative_samples:
        idx = rng.choice(len(neg_pool), size=negative_samples, replace=False)
        neg = [neg_pool[i] for i in idx]
    else:
        neg = neg_pool

    candidates = [chosen] + neg

    pos_base = position_features(board)
    out = []
    for mv in candidates:
        out.append({
            "game_id": row.game_id,
            "ply": int(row.ply),
            "fen": row.fen,
            "label": int(mv.uci() == chosen.uci()),
            "position_features": pos_base,
            "move_features": move_features(board, mv),
            "delta_features": delta_features(board, mv),
        })
    return out

In [14]:
# Enhanced move properties for detailed analysis of each candidate move.

def get_detailed_move_properties(board: chess.Board, move: chess.Move, is_chosen: bool) -> dict:
    """
    Extract comprehensive move properties for analysis.

    Parameters:
      board: chess.Board position before move
      move: chess.Move to analyze
      is_chosen: bool, whether this is the labeled ground-truth move

    Returns:
      dict of move properties
    """
    from_sq = move.from_square
    to_sq = move.to_square
    piece_on_from = board.piece_at(from_sq)
    piece_on_to = board.piece_at(to_sq)

    # Basic move properties.
    piece_type_char = piece_on_from.symbol().lower() if piece_on_from else '?'
    moving_piece_value = PIECE_VALUES.get(piece_on_from.piece_type, 0) if piece_on_from else 0

    # Move flags.
    is_capture = board.is_capture(move)
    is_check = board.gives_check(move)
    is_castling = board.is_castling(move)

    # Distance calculation.
    from_file = chess.square_file(from_sq)  # 0-7 (a-h)
    from_rank = chess.square_rank(from_sq)  # 0-7 (1-8 in chess notation)
    to_file = chess.square_file(to_sq)
    to_rank = chess.square_rank(to_sq)

    file_delta = to_file - from_file
    rank_delta = to_rank - from_rank

    # Euclidean distance (always positive).
    distance = np.sqrt(file_delta**2 + rank_delta**2)

    # Direction: positive if moving "forward" (towards opponent), negative if retreating.
    # For White: forward = increasing rank. For Black: forward = decreasing rank.
    if board.turn == chess.WHITE:
        direction_sign = 1 if rank_delta > 0 else (-1 if rank_delta < 0 else 0)
    else:
        direction_sign = -1 if rank_delta > 0 else (1 if rank_delta < 0 else 0)

    signed_distance = distance * direction_sign if direction_sign != 0 else distance

    # Analyze alternative captures from the same piece.
    # For the "avoids capture" feature, only count captures where the target piece
    # is equal or higher value than the moving piece.
    all_moves_from_square = [m for m in board.legal_moves if m.from_square == from_sq]
    capture_moves_from_square = [m for m in all_moves_from_square if board.is_capture(m)]
    equal_or_better_captures = []

    # Build list of capturable pieces (by value) that meet the threshold.
    capturable_pieces = []
    for cap_move in capture_moves_from_square:
        target_piece = board.piece_at(cap_move.to_square)
        if target_piece:
            target_value = PIECE_VALUES.get(target_piece.piece_type, 0)
            capturable_pieces.append(target_value)
            if target_value >= moving_piece_value:
                equal_or_better_captures.append(cap_move)

    # Summary of capture options: max capturable value, whether chosen move avoids a favorable capture.
    max_capturable = max(capturable_pieces) if capturable_pieces else 0
    piece_avoided_capture = (not is_capture) and len(equal_or_better_captures) > 0

    # Checkmate is a simple but useful move property for both positive and negative moves.
    next_board = board.copy(stack=False)
    next_board.push(move)
    delivers_checkmate = int(next_board.is_checkmate())

    return {
        "is_chosen_move": int(is_chosen),
        "piece_type": piece_type_char,
        "is_capture": int(is_capture),
        "is_check": int(is_check),
        "is_castling": int(is_castling),
        "delivers_checkmate": delivers_checkmate,
        "distance": float(distance),
        "signed_distance": float(signed_distance),
        "from_square": int(from_sq),
        "to_square": int(to_sq),
        "from_file": int(from_file),
        "from_rank": int(from_rank),
        "to_file": int(to_file),
        "to_rank": int(to_rank),
        "num_alt_captures_from_piece": int(len(capture_moves_from_square)),
        "num_equal_or_better_captures": int(len(equal_or_better_captures)),
        "max_capturable_value": int(max_capturable),
        "avoids_capture": int(piece_avoided_capture),
        "captured_piece_value": int(PIECE_VALUES.get(piece_on_to.piece_type, 0)) if piece_on_to else 0,
    }


def make_labeled_examples_v2(row, num_negatives: int = 10, rng_seed: int = 42):
    """
    Build training examples (1 positive + N negative samples) with enhanced move properties.
    The positive move is the actual chosen move; negatives are random legal moves.
    """
    board = chess.Board(row.fen)
    chosen_uci = row.chosen_move

    try:
        chosen_move = chess.Move.from_uci(chosen_uci)
    except Exception:
        return []

    if chosen_move not in board.legal_moves:
        return []

    # Get all legal moves and exclude the chosen one.
    legal_moves = list(board.legal_moves)
    other_legal = [m for m in legal_moves if m.uci() != chosen_uci]

    # Sample negatives reproducibly.
    rng = np.random.default_rng(rng_seed)
    if len(other_legal) > num_negatives:
        sampled_idx = rng.choice(len(other_legal), size=num_negatives, replace=False)
        negative_moves = [other_legal[i] for i in sampled_idx]
    else:
        negative_moves = other_legal

    # Build candidate list: chosen move is positive (label=1), others are negative (label=0).
    candidates = [(chosen_move, 1)] + [(m, 0) for m in negative_moves]

    examples = []
    for move, label in candidates:
        move_props = get_detailed_move_properties(board, move, bool(label))
        examples.append(
            {
                "game_id": row.game_id,
                "ply": int(row.ply),
                "fen": row.fen,
                **move_props,
            }
        )

    return examples

In [15]:
def extract_position_features(board: chess.Board) -> dict:
    """Extract all position features for both material and positional analysis."""
    color = board.turn
    enemy_color = not color
    
    # === Material & Piece Counts ===
    own_pieces = chess.popcount(board.occupied_co[color])
    enemy_pieces = chess.popcount(board.occupied_co[enemy_color])
    own_material = sum(len(board.pieces(pt, color)) * PIECE_VALUES[pt] for pt in PIECE_VALUES)
    enemy_material = sum(len(board.pieces(pt, enemy_color)) * PIECE_VALUES[pt] for pt in PIECE_VALUES)
    material_balance = own_material - enemy_material
    
    # === Pawn Structure Features ===
    own_pawns = list(board.pieces(chess.PAWN, color))
    own_pawns_set = set(own_pawns)  # For O(1) lookups in pawn chain checking
    enemy_pawns = list(board.pieces(chess.PAWN, enemy_color))
    
    # Helper: check if pawn on square can advance (is not blocked)
    def pawn_is_passed(sq: int, color: bool, enemy_pawns_list: list) -> bool:
        """A pawn is passed if no enemy pawn can stop it."""
        pawn_file = chess.square_file(sq)
        pawn_rank = chess.square_rank(sq)
        for ep in enemy_pawns_list:
            ep_file = chess.square_file(ep)
            ep_rank = chess.square_rank(ep)
            if abs(ep_file - pawn_file) <= 1:  # On same or adjacent file
                if color == chess.WHITE and ep_rank > pawn_rank:
                    return False  # Enemy pawn ahead blocks
                elif color == chess.BLACK and ep_rank < pawn_rank:
                    return False
        return True
    
    # Helper: check if pawn is backward
    def pawn_is_backward(sq: int, color: bool) -> bool:
        """A pawn is backward if it can't advance safely and has no friendly support."""
        pawn_file = chess.square_file(sq)
        pawn_rank = chess.square_rank(sq)
        
        # Try to move forward one square
        forward_offset = 8 if color == chess.WHITE else -8
        forward_sq = sq + forward_offset
        if not (0 <= forward_sq <= 63):
            return False  # Pawn is on promotion rank, not backward
        
        # Check if forward square is blocked
        if board.piece_at(forward_sq) is not None:
            blocked = True
        else:
            # Check if moving forward is safe (not attacked by enemy pawn)
            attacked = False
            for ep in enemy_pawns:
                ep_file = chess.square_file(ep)
                if abs(ep_file - pawn_file) == 1:
                    ep_rank = chess.square_rank(ep)
                    # Enemy pawn attacks diagonal squares in front of it
                    attack_offset = 8 if not color else -8  # Direction enemy pawns attack
                    if ep + attack_offset == forward_sq or ep + attack_offset + (1 if ep_file < pawn_file else -1) == forward_sq:
                        attacked = True
                        break
            blocked = attacked or board.piece_at(forward_sq) is not None
        
        if not blocked:
            return False  # Pawn can advance safely
        
        # Check if pawn has support on adjacent files
        support_left = pawn_file > 0 and chess.square(pawn_file - 1, pawn_rank) in own_pawns
        support_right = pawn_file < 7 and chess.square(pawn_file + 1, pawn_rank) in own_pawns
        
        return not (support_left or support_right)
    
    # Helper: find connected pawns (adjacent files at same rank)
    def pawns_connected(sq: int, all_own_pawns: list) -> bool:
        """Check if pawn at sq has a friendly pawn on adjacent file at same rank."""
        f = chess.square_file(sq)
        r = chess.square_rank(sq)
        left = chess.square(f - 1, r) if f > 0 else None
        right = chess.square(f + 1, r) if f < 7 else None
        return (left in all_own_pawns) or (right in all_own_pawns)
    
    # Helper: check if pawn is isolated (no pawns on adjacent files, any rank)
    def pawn_is_isolated(sq: int, all_own_pawns: list) -> bool:
        """Pawn is isolated if NO friendly pawns exist on the adjacent files (any rank)."""
        f = chess.square_file(sq)
        left_file = f - 1
        right_file = f + 1
        
        # Check if there are any pawns on the left or right files
        has_left_neighbor = any(chess.square_file(p) == left_file for p in all_own_pawns)
        has_right_neighbor = any(chess.square_file(p) == right_file for p in all_own_pawns)
        
        return not (has_left_neighbor or has_right_neighbor)
    
    # Helper: check if pawn is part of diagonal chain
    def pawn_in_chain(sq: int, all_own_pawns_set: set, color: bool) -> bool:
        """Pawn is in a chain if it has a friendly pawn on any diagonal neighbor.
        
        Uses bitboard-style diagonal checking: for each pawn, check all 4 diagonal
        neighbors (up-left, up-right, down-left, down-right) for a friendly pawn.
        """
        f = chess.square_file(sq)
        r = chess.square_rank(sq)
        # Check all 4 diagonals regardless of color direction
        diagonal_neighbors = [
            (f - 1, r + 1),  # up-left
            (f + 1, r + 1),  # up-right
            (f - 1, r - 1),  # down-left
            (f + 1, r - 1),  # down-right
        ]
        for nf, nr in diagonal_neighbors:
            if 0 <= nf <= 7 and 0 <= nr <= 7:
                neighbor_sq = chess.square(nf, nr)
                if neighbor_sq in all_own_pawns_set:
                    return True
        return False
    
    passed_pawns = sum(1 for sq in own_pawns if pawn_is_passed(sq, color, enemy_pawns))
    doubled_pawns = sum(1 for f in range(8) if sum(1 for sq in own_pawns if chess.square_file(sq) == f) >= 2)
    isolated_pawns = sum(1 for sq in own_pawns if pawn_is_isolated(sq, own_pawns))
    backward_pawns = sum(1 for sq in own_pawns if pawn_is_backward(sq, color))
    
    # Pawn islands: contiguous file groups with pawns
    pawn_files = sorted(set(chess.square_file(sq) for sq in own_pawns))
    pawn_islands = 1 if pawn_files else 0
    for i in range(1, len(pawn_files)):
        if pawn_files[i] - pawn_files[i-1] > 1:
            pawn_islands += 1
    
    connected_pawns = sum(1 for sq in own_pawns if pawns_connected(sq, own_pawns))
    chain_pawns = sum(1 for sq in own_pawns if pawn_in_chain(sq, own_pawns_set, color))
    
    # === Piece Mobility ===
    mobility_by_type = {}
    for pt in [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]:
        total_attacks = 0
        for sq in board.pieces(pt, color):
            attacks = set(board.attacks(sq))
            # For pawns, attacks are limited to captures (diagonal captures)
            if pt == chess.PAWN:
                attacks = {s for s in attacks if board.piece_at(s) is None or board.piece_at(s).color == enemy_color}
            else:
                attacks = {s for s in attacks if board.piece_at(s) is None or board.piece_at(s).color != color}
            total_attacks += len(attacks)
        pt_name = {chess.PAWN: "pawn", chess.KNIGHT: "knight", chess.BISHOP: "bishop", 
                   chess.ROOK: "rook", chess.QUEEN: "queen", chess.KING: "king"}[pt]
        mobility_by_type[pt_name] = total_attacks
    
    # === Rook Features ===
    own_rooks = [sq for sq in board.pieces(chess.ROOK, color)]
    rooks_on_open_files = 0
    rooks_on_semi_open = 0
    rooks_on_seventh = 0
    seventh_rank = 6 if color == chess.WHITE else 1
    
    for rook_sq in own_rooks:
        rook_file = chess.square_file(rook_sq)
        own_pawns_on_file = sum(1 for sq in own_pawns if chess.square_file(sq) == rook_file)
        enemy_pawns_on_file = sum(1 for sq in enemy_pawns if chess.square_file(sq) == rook_file)
        
        if own_pawns_on_file == 0 and enemy_pawns_on_file == 0:
            rooks_on_open_files += 1
        elif own_pawns_on_file == 0:
            rooks_on_semi_open += 1
        
        if chess.square_rank(rook_sq) == seventh_rank:
            rooks_on_seventh += 1
    
    # === Knight Outposts ===
    # Outpost: knight on advanced square not attacked by enemy pawns
    knight_outposts = 0
    for knight_sq in board.pieces(chess.KNIGHT, color):
        rank = chess.square_rank(knight_sq)
        if (color == chess.WHITE and rank >= 5) or (color == chess.BLACK and rank <= 2):
            # Check if attacked by enemy pawn
            attacked_by_pawn = False
            for ep in enemy_pawns:
                ep_file = chess.square_file(ep)
                knight_file = chess.square_file(knight_sq)
                if abs(knight_file - ep_file) == 1:
                    ep_rank = chess.square_rank(ep)
                    attack_rank = ep_rank + (8 if not color else -8)
                    if attack_rank == rank:
                        attacked_by_pawn = True
                        break
            if not attacked_by_pawn:
                knight_outposts += 1
    
    # === Pawn Shield (pawns in front of king) ===
    king_sq = board.king(color)
    pawn_shield = 0
    if king_sq is not None:
        king_file = chess.square_file(king_sq)
        king_rank = chess.square_rank(king_sq)
        shield_rank = king_rank + (1 if color == chess.WHITE else -1)
        for f in range(max(0, king_file - 1), min(8, king_file + 2)):
            shield_sq = chess.square(f, shield_rank)
            if shield_sq in own_pawns:
                pawn_shield += 1
    
    # === King Safety Features ===
    king_center_distance = 999
    if king_sq is not None:
        king_file = chess.square_file(king_sq)
        king_rank = chess.square_rank(king_sq)
        # Distance to center squares (d4, e4, d5, e5 are 27, 28, 35, 36)
        center_squares = [27, 28, 35, 36]
        for c_sq in center_squares:
            c_file = chess.square_file(c_sq)
            c_rank = chess.square_rank(c_sq)
            dist = max(abs(king_file - c_file), abs(king_rank - c_rank))
            king_center_distance = min(king_center_distance, dist)
    
    # King near open/semi-open files
    king_near_open = 0
    king_near_semi = 0
    if king_sq is not None:
        king_file = chess.square_file(king_sq)
        for f in range(max(0, king_file - 2), min(8, king_file + 3)):
            own_on_f = sum(1 for sq in own_pawns if chess.square_file(sq) == f)
            enemy_on_f = sum(1 for sq in enemy_pawns if chess.square_file(sq) == f)
            if own_on_f == 0 and enemy_on_f == 0:
                king_near_open += 1
            elif own_on_f == 0:
                king_near_semi += 1
    
    # === King Zone Attackers ===
    if king_sq is not None:
        king_file = chess.square_file(king_sq)
        king_rank = chess.square_rank(king_sq)
        king_zone = {chess.square(f, r) for f in range(max(0, king_file-1), min(8, king_file+2)) 
                     for r in range(max(0, king_rank-1), min(8, king_rank+2))}
        
        zone_attackers = 0
        zone_pawn_attackers = 0
        zone_knight_attackers = 0
        zone_bishop_attackers = 0
        zone_rook_attackers = 0
        zone_queen_attackers = 0
        
        for pt in [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]:
            for enemy_sq in board.pieces(pt, enemy_color):
                piece = board.piece_at(enemy_sq)
                attacks = set(board.attacks(enemy_sq))
                zone_hits = attacks & king_zone
                if zone_hits:
                    zone_attackers += 1
                    if piece.piece_type == chess.PAWN:
                        zone_pawn_attackers += 1
                    elif piece.piece_type == chess.KNIGHT:
                        zone_knight_attackers += 1
                    elif piece.piece_type == chess.BISHOP:
                        zone_bishop_attackers += 1
                    elif piece.piece_type == chess.ROOK:
                        zone_rook_attackers += 1
                    elif piece.piece_type == chess.QUEEN:
                        zone_queen_attackers += 1
    else:
        zone_attackers = zone_pawn_attackers = zone_knight_attackers = zone_bishop_attackers = zone_rook_attackers = zone_queen_attackers = 0
    
    # === King to King Distance ===
    king_to_enemy_king_dist = 999
    enemy_king_sq = board.king(enemy_color)
    if king_sq is not None and enemy_king_sq is not None:
        king_file = chess.square_file(king_sq)
        king_rank = chess.square_rank(king_sq)
        enemy_king_file = chess.square_file(enemy_king_sq)
        enemy_king_rank = chess.square_rank(enemy_king_sq)
        king_to_enemy_king_dist = max(abs(king_file - enemy_king_file), abs(king_rank - enemy_king_rank))
    
    # === King to Pawn Distances ===
    king_to_friend_pawn_dist = 999
    king_to_enemy_pawn_dist = 999
    if king_sq is not None:
        king_file = chess.square_file(king_sq)
        king_rank = chess.square_rank(king_sq)
        for sq in own_pawns:
            f = chess.square_file(sq)
            r = chess.square_rank(sq)
            dist = max(abs(king_file - f), abs(king_rank - r))
            king_to_friend_pawn_dist = min(king_to_friend_pawn_dist, dist)
        for sq in enemy_pawns:
            f = chess.square_file(sq)
            r = chess.square_rank(sq)
            dist = max(abs(king_file - f), abs(king_rank - r))
            king_to_enemy_pawn_dist = min(king_to_enemy_pawn_dist, dist)
    
    # === Piece Tropism (inverse distance to king) ===
    king_tropism = 0.0
    if king_sq is not None:
        king_file = chess.square_file(king_sq)
        king_rank = chess.square_rank(king_sq)
        for pt in [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]:
            for enemy_sq in board.pieces(pt, enemy_color):
                f = chess.square_file(enemy_sq)
                r = chess.square_rank(enemy_sq)
                dist = max(abs(king_file - f), abs(king_rank - r))
                if dist > 0:
                    king_tropism += 1.0 / dist
    
    # === Control Features ===
    controlled_squares = set()
    space_in_enemy_half_squares = set()
    center_control_squares = set()
    enemy_controlled_squares = set()
    center_squares = {27, 28, 35, 36}  # d4, e4, d5, e5
    for pt in [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]:
        for own_sq in board.pieces(pt, color):
            attacks = set(board.attacks(own_sq))
            controlled_squares.update(attacks)
            for attacked_sq in attacks:
                if (color == chess.WHITE and chess.square_rank(attacked_sq) >= 4) or \
                   (color == chess.BLACK and chess.square_rank(attacked_sq) <= 3):
                    space_in_enemy_half_squares.add(attacked_sq)
                if attacked_sq in center_squares:
                    center_control_squares.add(attacked_sq)
    
    
    for pt in [chess.PAWN, chess.KNIGHT, chess.BISHOP, chess.ROOK, chess.QUEEN, chess.KING]:
        for enemy_sq in board.pieces(pt, enemy_color):
            attacks = set(board.attacks(enemy_sq))
            enemy_controlled_squares.update(attacks)
    
    return {
        "own_piece_count": own_pieces,
        "enemy_piece_count": enemy_pieces,
        "material_balance": material_balance,
        "turn_white": int(color),
        "pawn_mobility": mobility_by_type.get("pawn", 0),
        "knight_mobility": mobility_by_type.get("knight", 0),
        "bishop_mobility": mobility_by_type.get("bishop", 0),
        "rook_mobility": mobility_by_type.get("rook", 0),
        "queen_mobility": mobility_by_type.get("queen", 0),
        "king_mobility": mobility_by_type.get("king", 0),
        "rook_open_file_count": rooks_on_open_files,
        "rook_semi_open_file_count": rooks_on_semi_open,
        "rook_seventh_rank_count": rooks_on_seventh,
        "knight_outpost_count": knight_outposts,
        "passed_pawn_count": passed_pawns,
        "doubled_pawn_file_count": doubled_pawns,
        "isolated_pawn_count": isolated_pawns,
        "backward_pawn_count": backward_pawns,
        "pawn_island_count": pawn_islands,
        "connected_pawn_count": connected_pawns,
        "pawn_chain_count": chain_pawns,
        "pawn_shield_count": pawn_shield,
        "king_center_distance": king_center_distance,
        "king_near_open_file_count": king_near_open,
        "king_near_semi_open_file_count": king_near_semi,
        "enemy_king_zone_attackers": zone_attackers,
        "enemy_king_zone_pawn_attackers": zone_pawn_attackers,
        "enemy_king_zone_knight_attackers": zone_knight_attackers,
        "enemy_king_zone_bishop_attackers": zone_bishop_attackers,
        "enemy_king_zone_rook_attackers": zone_rook_attackers,
        "enemy_king_zone_queen_attackers": zone_queen_attackers,
        "king_to_enemy_king_distance": king_to_enemy_king_dist,
        "king_to_friendly_pawn_distance": king_to_friend_pawn_dist,
        "king_to_enemy_pawn_distance": king_to_enemy_pawn_dist,
        "king_tropism": king_tropism,
        "controlled_square_count": len(controlled_squares),
        "space_in_enemy_half": len(space_in_enemy_half_squares),
        "center_control_count": len(center_control_squares),
        "enemy_controlled_square_count": len(enemy_controlled_squares),
    }


def _numeric_delta(before_dict: dict, after_dict: dict) -> dict:
    """Compute delta features: after - before for numeric values."""
    return {f"delta_position_{k}": after_dict[k] - before_dict[k] for k in before_dict if isinstance(before_dict[k], (int, float))}


def _prefix_dict(d: dict, prefix: str) -> dict:
    """Add prefix to all keys in dict."""

    return {f"{prefix}{k}": v for k, v in d.items()}


In [16]:
def process_unprocessed_dataframe(unprocessed_df: pd.DataFrame) -> pd.DataFrame:
    """Compute expensive position features for each candidate row.

    We compute features for both the current position and the resulting next
    position, then derive delta features between the two.
    """
    processed_rows = []
    
    # Move feature columns to include from unprocessed rows (excluding unnecessary ones)
    move_feature_cols = [
        "is_chosen_move", "piece_type", "is_capture", "is_check", "is_castling",
        "delivers_checkmate", "signed_distance", "avoids_capture", "captured_piece_value"
    ]
    
    for row in unprocessed_df.itertuples(index=False):
        current_board = chess.Board(row.current_fen)
        next_board = chess.Board(row.next_fen)

        # Current position features are kept; delta features show the effect of the move.
        # We skip next_position_features since delta features capture the change.
        current_position_features = extract_position_features(current_board)
        next_position_features = extract_position_features(next_board)
        delta_position_features = _numeric_delta(current_position_features, next_position_features)

        # Collect move features from the unprocessed row
        move_features_dict = {}
        for col in move_feature_cols:
            if hasattr(row, col):
                move_features_dict[col] = getattr(row, col)

        processed_rows.append(
            {
                "game_id": row.game_id,
                "ply": int(row.ply),
                "move_uci": row.move_uci,
                "current_fen": row.current_fen,
                "next_fen": row.next_fen,
                **move_features_dict,
                **_prefix_dict(current_position_features, "position_"),
                **delta_position_features,
            }
        )

    return pd.DataFrame(processed_rows)

## Production pipeline

Run the notebook in two stages: first build the compact unprocessed candidate-move CSV from the known games file, then process that CSV in batches into the final feature table.

The unprocessed file stores only `game_id`, `date`, `fen`, `move_played`, and `is_chosen_move`. Stage 2 uses those rows to compute candidate-move features, current-position features, and delta features.

In [17]:
# Stage 1: build the unprocessed candidate-move file.
NEGATIVE_MOVE_SAMPLES = 10
BATCH_SIZE = 1000
UNPROCESSED_CSV_PATH = Path("game-data") / "training_unprocessed.csv"

import csv


def stream_unprocessed_csv(positions_df: pd.DataFrame, unprocessed_csv_path: Path, num_negative_moves: int = 10, rng_seed: int = 42) -> Path:
    unprocessed_csv_path = Path(unprocessed_csv_path)
    unprocessed_csv_path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = ["game_id", "date", "ply", "fen", "move_played", "is_chosen_move"]
    total_rows = 0

    with unprocessed_csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()

        for idx, row in enumerate(positions_df.itertuples(index=False), 1):
            board = chess.Board(row.fen)
            chosen_uci = row.chosen_move

            try:
                chosen_move = chess.Move.from_uci(chosen_uci)
            except Exception:
                continue

            if chosen_move not in board.legal_moves:
                continue

            legal_moves = list(board.legal_moves)
            other_legal = [move for move in legal_moves if move.uci() != chosen_uci]

            rng = np.random.default_rng(rng_seed + idx)
            if len(other_legal) > num_negative_moves:
                sampled_idx = rng.choice(len(other_legal), size=num_negative_moves, replace=False)
                candidate_moves = [chosen_move] + [other_legal[i] for i in sampled_idx]
            else:
                candidate_moves = [chosen_move] + other_legal

            for move in candidate_moves:
                writer.writerow({
                    "game_id": row.game_id,
                    "date": row.date,
                    "ply": int(row.ply),
                    "fen": row.fen,
                    "move_played": move.uci(),
                    "is_chosen_move": int(move.uci() == chosen_uci),
                })
                total_rows += 1

            if idx % 100 == 0:
                print(f"  Processed {idx:,} positions -> {total_rows:,} candidate rows written")

    print(f"Saved unprocessed candidate rows to {unprocessed_csv_path}")
    print(f"Unprocessed row count: {total_rows:,}")
    return unprocessed_csv_path


def write_unprocessed_csv(positions_df: pd.DataFrame, unprocessed_csv_path: Path, num_negative_moves: int = 10, rng_seed: int = 42) -> Path:
    return stream_unprocessed_csv(positions_df, unprocessed_csv_path, num_negative_moves=num_negative_moves, rng_seed=rng_seed)

unprocessed_csv_path = write_unprocessed_csv(my_positions, UNPROCESSED_CSV_PATH, num_negative_moves=NEGATIVE_MOVE_SAMPLES, rng_seed=42)

  Processed 100 positions -> 1,042 candidate rows written
  Processed 200 positions -> 2,142 candidate rows written
  Processed 300 positions -> 3,201 candidate rows written
  Processed 400 positions -> 4,293 candidate rows written
  Processed 500 positions -> 5,379 candidate rows written
  Processed 600 positions -> 6,413 candidate rows written
  Processed 700 positions -> 7,480 candidate rows written
  Processed 800 positions -> 8,512 candidate rows written
  Processed 900 positions -> 9,573 candidate rows written
  Processed 1,000 positions -> 10,656 candidate rows written
  Processed 1,100 positions -> 11,733 candidate rows written
  Processed 1,200 positions -> 12,776 candidate rows written
  Processed 1,300 positions -> 13,826 candidate rows written
  Processed 1,400 positions -> 14,820 candidate rows written
  Processed 1,500 positions -> 15,662 candidate rows written
  Processed 1,600 positions -> 16,726 candidate rows written
  Processed 1,700 positions -> 17,781 candidate row

## Batch processing

Load the saved unprocessed CSV and convert it to the final processed feature table in chunks.

To resume a partial run, set `RESUME_PROCESSED_CSV_PATH` to an existing processed CSV path before running the next cell. The notebook will use the row count from that file and continue from the matching position in the unprocessed CSV.

In [18]:
# Stage 2: load the unprocessed CSV and write the processed feature table in batches.
DEFAULT_PROCESSED_CSV_PATH = Path("game-data") / "training_processed.csv"
RESUME_PROCESSED_CSV_PATH = None  # set this to an existing processed CSV to resume from that file
PROCESSED_BATCH_SIZE = 1000


def process_unprocessed_dataframe(unprocessed_df: pd.DataFrame) -> pd.DataFrame:
    """Compute candidate-move, current-position, and delta features for each row."""
    processed_rows = []

    for row in unprocessed_df.itertuples(index=False):
        current_board = chess.Board(row.fen)
        move = chess.Move.from_uci(row.move_played)
        if move not in current_board.legal_moves:
            continue

        next_board = current_board.copy(stack=False)
        next_board.push(move)

        current_position_features = extract_position_features(current_board)
        next_position_features = extract_position_features(next_board)
        delta_position_features = _numeric_delta(current_position_features, next_position_features)

        candidate_move_features = get_detailed_move_properties(current_board, move, bool(row.is_chosen_move))
        candidate_move_features.pop("is_chosen_move", None)

        processed_row = {
            "game_id": row.game_id,
            "date": row.date,
            "ply": int(row.ply),
            "move_uci": row.move_played,
            "current_fen": row.fen,
            "next_fen": next_board.fen(),
            "is_chosen_move": int(row.is_chosen_move),
            **candidate_move_features,
            **{f"position_{k}": v for k, v in current_position_features.items()},
            **{f"next_position_{k}": v for k, v in next_position_features.items()},
            **delta_position_features,
        }
        processed_rows.append(processed_row)

    return pd.DataFrame(processed_rows)


def process_unprocessed_csv(
    unprocessed_csv_path: Path,
    processed_csv_path: Path,
    batch_size: int = 1000,
    resume_processed_csv_path: Path | None = None,
) -> Path:
    unprocessed_csv_path = Path(unprocessed_csv_path)
    processed_csv_path = Path(processed_csv_path)
    output_path = processed_csv_path
    processed_rows = 0

    if resume_processed_csv_path is not None:
        resume_processed_csv_path = Path(resume_processed_csv_path)
        if resume_processed_csv_path.exists():
            output_path = resume_processed_csv_path
            processed_rows = len(pd.read_csv(output_path))
            print(f"Resuming from {processed_rows:,} processed rows in {output_path}")
    elif processed_csv_path.exists():
        output_path = processed_csv_path
        processed_rows = len(pd.read_csv(output_path))
        print(f"Continuing from {processed_rows:,} processed rows in {output_path}")

    total_input_rows = sum(1 for _ in unprocessed_csv_path.open("r", encoding="utf-8")) - 1
    if processed_rows >= total_input_rows:
        print(f"No rows left to process. Unprocessed rows: {total_input_rows:,}")
        return output_path

    total_output_rows = processed_rows
    next_log_row = ((total_output_rows // 100) + 1) * 100

    reader = pd.read_csv(
        unprocessed_csv_path,
        chunksize=batch_size,
        skiprows=range(1, processed_rows + 1) if processed_rows else None,
    )

    input_row_start = processed_rows
    for batch_df in reader:
        input_row_end = min(input_row_start + len(batch_df), total_input_rows)
        processed_batch = process_unprocessed_dataframe(batch_df)
        write_header = not output_path.exists() or output_path.stat().st_size == 0
        processed_batch.to_csv(
            output_path,
            mode="a" if output_path.exists() else "w",
            header=write_header,
            index=False,
        )
        total_output_rows += len(processed_batch)
        print(f"Processed rows {input_row_start:,}-{input_row_end - 1:,} -> {len(processed_batch):,} output rows")
        while next_log_row <= total_output_rows:
            print(f"  ✓ {next_log_row:,} total rows written")
            next_log_row += 100
        input_row_start = input_row_end

    return output_path

processed_csv_path = process_unprocessed_csv(
    UNPROCESSED_CSV_PATH,
    DEFAULT_PROCESSED_CSV_PATH,
    batch_size=PROCESSED_BATCH_SIZE,
    resume_processed_csv_path=RESUME_PROCESSED_CSV_PATH,
)
print(f"Processed CSV saved to: {processed_csv_path}")

Processed rows 0-999 -> 1,000 output rows
  ✓ 100 total rows written
  ✓ 200 total rows written
  ✓ 300 total rows written
  ✓ 400 total rows written
  ✓ 500 total rows written
  ✓ 600 total rows written
  ✓ 700 total rows written
  ✓ 800 total rows written
  ✓ 900 total rows written
  ✓ 1,000 total rows written
Processed rows 1,000-1,999 -> 1,000 output rows
  ✓ 1,100 total rows written
  ✓ 1,200 total rows written
  ✓ 1,300 total rows written
  ✓ 1,400 total rows written
  ✓ 1,500 total rows written
  ✓ 1,600 total rows written
  ✓ 1,700 total rows written
  ✓ 1,800 total rows written
  ✓ 1,900 total rows written
  ✓ 2,000 total rows written
Processed rows 2,000-2,999 -> 1,000 output rows
  ✓ 2,100 total rows written
  ✓ 2,200 total rows written
  ✓ 2,300 total rows written
  ✓ 2,400 total rows written
  ✓ 2,500 total rows written
  ✓ 2,600 total rows written
  ✓ 2,700 total rows written
  ✓ 2,800 total rows written
  ✓ 2,900 total rows written
  ✓ 3,000 total rows written
Processed 